# UC1 - Proactive Failure Risk Detection in SD-WAN Devices

This notebook aims to explore whether operational telemetry from network devices can be used to identify behavioral patterns and predict failure risk. Using a synthetic dataset of ISR1000 routers, it first applies unsupervised learning to discover natural groupings of devices based on metrics such as CPU, memory, temperature, and fan speed. These clusters are then analyzed to understand their relationship with observed failure behavior.

Building on these insights, the notebook tests a supervised learning approach to determine if device risk levels (low, medium, high) can be predicted in advance from operational features, and validates the model by classifying both known and newly simulated devices.

In [ ]:
import sys
path_files ='Files'

In [ ]:
from os.path import join
import pandas as pd

# Data Collection



In [ ]:
df=pd.read_csv(join(path_files,'ISR1000_sintetico.csv'),index_col=0)

In [ ]:
df

# **Data Normalization**

In [ ]:
from sklearn.preprocessing import StandardScaler
# Features a usar para clustering
features = ["Operating_Temperature_Nominal",
            "Operating_Humidity_Nominal",
            "Fan_Speed",
            "CPU",
            "Memory"]

X = df[features].values

# Escalado
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# **1 - PCA**

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# Reducir a 2D con PCA
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

# Graficar todos los puntos juntos
plt.figure(figsize=(7,6))
plt.scatter(X_pca[:,0], X_pca[:,1], color="steelblue", alpha=0.6, s=20)

plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("PCA 2D - ISR1000 Devices")
plt.show()

# **2 - Clustering**

> Finding the optimal number of clusters K



In [ ]:
# --- Elbow + Silhouette para elegir K ---
import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Usa tu matriz estandarizada:
# X_scaled = scaler.fit_transform(X)

Ks = range(2, 11)  # prueba K=2..10
inertias = []
silhouettes = []

for k in Ks:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    sil_score = silhouette_score(X_scaled, labels)
    silhouettes.append(sil_score)

# Plot Elbow (Inertia)
plt.figure(figsize=(6,4))
plt.plot(Ks, inertias, marker='o')
plt.xlabel('Número de clusters (K)')
plt.ylabel('Inercia (Suma de distancias cuadráticas)')
plt.title('Elbow Method')
plt.grid(True)
plt.show()

# Plot Silhouette
plt.figure(figsize=(6,4))
plt.plot(Ks, silhouettes, marker='o')
plt.xlabel('Número de clusters (K)')
plt.ylabel('Silhouette score')
plt.title('Silhouette vs K')
plt.grid(True)
plt.show()

# Resultado rápido
best_k_sil = Ks[int(np.argmax(silhouettes))]
print(f"Mejor K por silhouette ≈ {best_k_sil} (score={max(silhouettes):.3f})")


***Apply KMeans with 4 clusters***





In [ ]:
# 5) KMeans con 4 clusters (hipótesis original)
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df["Cluster"] = kmeans.fit_predict(X_scaled)

# 6) Calcular % de fallos en cada cluster
failure_rates = df.groupby("Cluster")["Failure_Last6Months"].mean() * 100

print(" Porcentaje de fallos por cluster:")
print(failure_rates)

# 7) Gráfico: distribución de fallos por cluster
failure_rates.plot(kind="bar", color="crimson", figsize=(6,4))
plt.ylabel("% de fallos")
plt.title("Tasa de fallos por cluster")
plt.show()

# 8) Gráfico PCA 2D coloreado por cluster

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(7,6))
for k in range(4):
    cluster_points = X_pca[df["Cluster"]==k]
    plt.scatter(cluster_points[:,0], cluster_points[:,1], label=f"Cluster {k}", alpha=0.6)

plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Clusters KMeans (PCA 2D)")
plt.legend()
plt.show()


# **3 - Post Clustering Analysis**

In [ ]:
# Calcular medias de todos los features por cluster
cluster_means = (
    df.groupby("Cluster")[[
        "Operating_Temperature_Nominal",
        "Operating_Humidity_Nominal",
        "Fan_Speed",
        "CPU",
        "Memory",
        "Failure_Last6Months"
    ]]
    .mean()
    .round(2)
    .rename(columns={
        "Operating_Temperature_Nominal": "Temp(°C)",
        "Operating_Humidity_Nominal": "Hum(%)",
        "Fan_Speed": "Fan(RPM)",
        "CPU": "CPU(%)",
        "Memory": "Mem(%)",
        "Failure_Last6Months": "Fail_rate"
    })
)

# Convertir tasa de fallos a porcentaje
cluster_means["Fail_rate"] = (cluster_means["Fail_rate"] * 100).round(2)

cluster_means


In [ ]:
# 9) Cantidad y porcentaje de equipos por cluster
cluster_counts = df["Cluster"].value_counts().sort_index()
cluster_percentages = (cluster_counts / len(df)) * 100

print("Distribución de equipos por cluster:")
print(pd.DataFrame({"Cantidad": cluster_counts, "%": cluster_percentages.round(2)}))

# 10) Gráfico de barras con cantidades y porcentajes
plt.figure(figsize=(6,4))
bars = plt.bar(cluster_counts.index, cluster_counts.values, color="steelblue", alpha=0.8)

# Etiquetas con % encima de cada barra
for bar, pct in zip(bars, cluster_percentages):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
             f"{pct:.1f}%", ha="center", va="bottom", fontsize=10, fontweight="bold")

plt.ylabel("Cantidad de equipos")
plt.xlabel("Cluster")
plt.title("Distribución de equipos por cluster")
plt.xticks(cluster_counts.index)
plt.show()


In [ ]:
import os
output_path = os.path.join(path_files, "ISR1000_dataset_labeled_clusters&risks.csv")
df.to_csv(output_path, index=False)
print(f"✅ Dataset guardado en: {output_path}")

# **4 - Classification Model**


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# 1) Mapping risks
risk_map = {2: "Alta", 0: "Media", 1: "Media", 3: "Baja"}
df["Risk_Label"] = df["Cluster"].map(risk_map)

# 2) Features y target
features = ["Operating_Temperature_Nominal",
            "Operating_Humidity_Nominal",
            "Fan_Speed",
            "CPU",
            "Memory"]

X = df[features].values
y = df["Risk_Label"].values

# 3) Split train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

# 4) Clasificador
clf = RandomForestClassifier(n_estimators=200, random_state=42)
clf.fit(X_train, y_train)




In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import seaborn as sns


# 1) Predicciones
y_pred = clf.predict(X_test)

# 2) Accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"🎯 Accuracy: {accuracy:.2f}")

# 3) Reporte detallado
print("\n📊 Reporte de Clasificación:")
print(classification_report(y_test, y_pred))

# 4) Matriz de confusión
cm = confusion_matrix(y_test, y_pred, labels=clf.classes_)

plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=clf.classes_, yticklabels=clf.classes_)
plt.xlabel("Predicción")
plt.ylabel("Real")
plt.title("Matriz de Confusión")
plt.show()


# ***Analyze new Devices***


In [ ]:
features = ["Operating_Temperature_Nominal",
            "Operating_Humidity_Nominal",
            "Fan_Speed",
            "CPU",
            "Memory"]
new_data = pd.DataFrame({
    "Operating_Temperature_Nominal": np.random.uniform(30, 60, 20),
    "Operating_Humidity_Nominal": np.random.uniform(40, 70, 20),
    "Fan_Speed": np.random.uniform(7000, 8000, 20),
    "CPU": np.random.uniform(30, 80, 20),
    "Memory": np.random.uniform(30, 80, 20),
})

In [ ]:
new_data

In [ ]:
# Class Predictions
y_new_pred = clf.predict(new_data[features])

# Add a risk level column to the dataset
new_data["Predicted_Risk"] = y_new_pred

print(new_data)


In [ ]:
# Device count by risk classification
risk_counts = new_data["Predicted_Risk"].value_counts()

# Bar chart showing the distribution
plt.figure(figsize=(6,4))
sns.barplot(x=risk_counts.index, y=risk_counts.values, palette="Set2")
plt.title("Distribución de Riesgo en los 20 equipos")
plt.xlabel("Nivel de Riesgo")
plt.ylabel("Cantidad de equipos")
plt.show()